# Анализ пользовательской оценки аудиокомментариев

Мини-опрос используется как качественная проверка финальной аудиодорожки. Участникам предлагается прослушать фрагмент/матч с автоматически сгенерированным тифлокомментарием и поставить одну из трёх оценок:

- **отлично** — комментарий понятный, уместный и помогает следить за эпизодом;
- **нормально** — в целом понятно, но есть отдельные шероховатости;
- **плохо** — комментарий мешает, путает или недостаточно информативен.

Также фиксируется, является ли участник футбольным болельщиком.


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

OUT_DIR = Path('outputs/survey_audio_feedback')
FIG_DIR = OUT_DIR / 'figures'
TAB_DIR = OUT_DIR / 'tables'
for d in [OUT_DIR, FIG_DIR, TAB_DIR]:
    d.mkdir(parents=True, exist_ok=True)

plt.rcParams.update({
    'font.family': 'DejaVu Sans',
    'axes.titlesize': 18,
    'axes.titleweight': 'bold',
    'axes.labelsize': 12,
    'xtick.labelsize': 11,
    'ytick.labelsize': 11,
})

HSE_BLUE = '#005BBB'
HSE_LIGHT = '#D7ECFF'
HSE_MID = '#5AA9E6'
HSE_DARK = '#0F2F4F'
GOOD = '#005BBB'
OK = '#5AA9E6'
BAD = '#B8C2CC'
TEXT = '#1F2937'
GRID = '#D9E2EC'


## 1. Данные опроса

Заполни `survey_rows` реальными 12 ответами. Поля:

- `respondent_id` — номер участника;
- `is_football_fan` — `1`, если человек регулярно смотрит футбол, иначе `0`;
- `rating` — `отлично`, `нормально` или `плохо`;
- `comment` — короткий свободный комментарий участника.


In [ ]:
survey_rows = [
    {'respondent_id': 1,  'is_football_fan': 1, 'rating': 'нормально', 'comment': ''},
    {'respondent_id': 2,  'is_football_fan': 1, 'rating': 'нормально', 'comment': ''},
    {'respondent_id': 3,  'is_football_fan': 1, 'rating': 'отлично',  'comment': ''},
    {'respondent_id': 4,  'is_football_fan': 0, 'rating': 'нормально', 'comment': ''},
    {'respondent_id': 5,  'is_football_fan': 0, 'rating': 'нормально', 'comment': ''},
    {'respondent_id': 6,  'is_football_fan': 0, 'rating': 'отлично',  'comment': ''},
    {'respondent_id': 7,  'is_football_fan': 0, 'rating': 'плохо',    'comment': ''},
    {'respondent_id': 8,  'is_football_fan': 0, 'rating': 'нормально', 'comment': ''},
    {'respondent_id': 9,  'is_football_fan': 1, 'rating': 'плохо',    'comment': ''},
    {'respondent_id': 10, 'is_football_fan': 0, 'rating': 'нормально', 'comment': ''},
    {'respondent_id': 11, 'is_football_fan': 0, 'rating': 'отлично',  'comment': ''},
    {'respondent_id': 12, 'is_football_fan': 0, 'rating': 'нормально', 'comment': ''},
]

survey_df = pd.DataFrame(survey_rows)
rating_order = ['отлично', 'нормально', 'плохо']
survey_df['rating'] = pd.Categorical(survey_df['rating'], categories=rating_order, ordered=True)
survey_df['group'] = np.where(survey_df['is_football_fan'].astype(int) == 1, 'Болельщики', 'Не болельщики')

assert len(survey_df) == 12, f'Ожидалось 12 ответов, сейчас: {len(survey_df)}'
assert survey_df['rating'].notna().all(), 'В rating должны быть только: отлично / нормально / плохо'

display(survey_df)
survey_df.to_csv(TAB_DIR / 'survey_responses_12.csv', index=False)


## 2. Сводная статистика


In [ ]:
summary_total = survey_df.groupby('rating', observed=False).size().reset_index(name='n')
summary_total['percent'] = summary_total['n'] / summary_total['n'].sum() * 100

summary_group = survey_df.groupby(['group', 'rating'], observed=False).size().reset_index(name='n')
summary_group['group_total'] = summary_group.groupby('group')['n'].transform('sum')
summary_group['percent'] = summary_group['n'] / summary_group['group_total'] * 100

fan_stats = survey_df.groupby('group').size().reset_index(name='n')
fan_stats['percent'] = fan_stats['n'] / fan_stats['n'].sum() * 100

print('Всего участников:', len(survey_df))
display(fan_stats)
display(summary_total)
display(summary_group)

summary_total.to_csv(TAB_DIR / 'rating_summary_total.csv', index=False)
summary_group.to_csv(TAB_DIR / 'rating_summary_by_group.csv', index=False)
fan_stats.to_csv(TAB_DIR / 'respondent_groups.csv', index=False)


## 3. Круговые диаграммы в стиле ВШЭ


In [ ]:
def autopct_count(values):
    total = sum(values)
    def _fmt(pct):
        count = int(round(pct * total / 100.0))
        return f'{pct:.0f}%
({count})'
    return _fmt

fig, axes = plt.subplots(1, 2, figsize=(13, 6.2))
fig.patch.set_facecolor('white')

vals = fan_stats['n'].to_numpy()
labels = fan_stats['group'].tolist()
colors_groups = [HSE_BLUE if x == 'Болельщики' else HSE_LIGHT for x in labels]
axes[0].pie(
    vals,
    labels=labels,
    colors=colors_groups,
    startangle=90,
    counterclock=False,
    autopct=autopct_count(vals),
    textprops={'fontsize': 12, 'color': TEXT, 'fontweight': 'bold'},
    wedgeprops={'linewidth': 1.5, 'edgecolor': 'white'},
)
axes[0].set_title('Состав участников опроса', color=TEXT, pad=12)

vals = summary_total['n'].to_numpy()
labels = summary_total['rating'].astype(str).tolist()
colors_rating = [GOOD, OK, BAD]
axes[1].pie(
    vals,
    labels=labels,
    colors=colors_rating,
    startangle=90,
    counterclock=False,
    autopct=autopct_count(vals),
    textprops={'fontsize': 12, 'color': TEXT, 'fontweight': 'bold'},
    wedgeprops={'linewidth': 1.5, 'edgecolor': 'white'},
)
axes[1].set_title('Общая оценка комментариев', color=TEXT, pad=12)

fig.suptitle('Пользовательская оценка аудиокомментариев', fontsize=21, fontweight='bold', color=TEXT, y=1.04)
plt.tight_layout()

out = FIG_DIR / 'survey_pie_charts.png'
fig.savefig(out, dpi=220, bbox_inches='tight', facecolor='white')
plt.show()
print('saved:', out)


## 4. Оценки по группам


In [ ]:
plot_group = summary_group.copy()
plot_group['rating'] = plot_group['rating'].astype(str)

fig, ax = plt.subplots(figsize=(11, 6.4))
fig.patch.set_facecolor('white')
ax.set_facecolor('white')

pivot = plot_group.pivot(index='group', columns='rating', values='percent').fillna(0)
pivot = pivot.reindex(['Болельщики', 'Не болельщики'])
pivot = pivot[rating_order]

bottom = np.zeros(len(pivot))
colors = {'отлично': GOOD, 'нормально': OK, 'плохо': BAD}

for rating in rating_order:
    vals = pivot[rating].to_numpy()
    bars = ax.bar(pivot.index, vals, bottom=bottom, label=rating, color=colors[rating], edgecolor='white', linewidth=1.2)
    for i, (bar, val) in enumerate(zip(bars, vals)):
        if val >= 7:
            ax.text(bar.get_x() + bar.get_width() / 2, bottom[i] + val / 2, f'{val:.0f}%', ha='center', va='center', fontsize=12, fontweight='bold', color='white' if rating != 'плохо' else TEXT)
    bottom += vals

ax.set_title('Распределение оценок по группам участников', color=TEXT, pad=16)
ax.set_ylabel('Доля внутри группы, %')
ax.set_ylim(0, 100)
ax.grid(axis='y', color=GRID, alpha=0.8)
ax.spines[['top', 'right']].set_visible(False)
ax.spines[['left', 'bottom']].set_color('#B8C2CC')
ax.legend(title='Оценка', frameon=False, ncol=3, loc='upper center', bbox_to_anchor=(0.5, -0.08))

plt.tight_layout()
out = FIG_DIR / 'survey_rating_by_group.png'
fig.savefig(out, dpi=220, bbox_inches='tight', facecolor='white')
plt.show()
print('saved:', out)


## 5. Свободные комментарии участников


In [ ]:
comments_only = survey_df[survey_df['comment'].fillna('').str.strip().ne('')].copy()
comments_only = comments_only[['respondent_id', 'group', 'rating', 'comment']]
display(comments_only)
comments_only.to_csv(TAB_DIR / 'survey_free_comments.csv', index=False)

if len(comments_only):
    print('Примеры комментариев для текста работы:')
    for _, r in comments_only.iterrows():
        print(f"- {r['group']}, оценка: {r['rating']}. {r['comment']}")
else:
    print('Пока комментарии не заполнены. Добавь реальные ответы в survey_rows.')


## 6. Текстовый вывод для ВКР


In [ ]:
n_total = len(survey_df)
n_fans = int((survey_df['group'] == 'Болельщики').sum())
n_nonfans = n_total - n_fans
rating_counts = summary_total.set_index('rating')['n'].to_dict()
excellent = int(rating_counts.get('отлично', 0))
ok = int(rating_counts.get('нормально', 0))
bad = int(rating_counts.get('плохо', 0))
positive = excellent + ok
positive_share = positive / n_total * 100 if n_total else 0

text = (
    f'В пользовательском мини-опросе приняли участие {n_total} человек: '
    f'{n_fans} футбольных болельщиков и {n_nonfans} участников, не относящих себя к регулярным зрителям футбола. '
    f'По итогам прослушивания аудиокомментария {excellent} участников оценили результат как «отлично», '
    f'{ok} — как «нормально», {bad} — как «плохо». '
    f'Таким образом, доля положительных и нейтрально-положительных оценок («отлично» или «нормально») составила {positive_share:.1f}%. '
    'Полученные ответы показывают, что автоматически сгенерированный комментарий в целом воспринимается как понятный, '
    'однако отдельные замечания участников могут быть использованы для доработки стиля и детализации описаний.'
)

print(text)
(OUT_DIR / 'survey_summary_text.txt').write_text(text, encoding='utf-8')
